# LedgerAI — Transaction Analysis Notebook
**Author:** Zohaib Ghani — BBA Accounting, Wilfrid Laurier University  
**Purpose:** Demonstrates how accounting transaction data is analyzed using Python (pandas) and SQL (SQLite).  
This notebook mirrors the analysis logic inside LedgerAI's web interface, but written in Python as it would be used in a real accounting firm's data pipeline.

---
### What this notebook does
1. Loads a transaction CSV (exported from LedgerAI or any accounting system)
2. Stores it in a SQLite database — a real relational database
3. Runs 10 accounting analysis queries using both **pandas** and **SQL**
4. Produces charts and a summary report

### Libraries used
- **pandas** — the standard Python library for data analysis. Every data analyst at an accounting firm uses this.
- **sqlite3** — built into Python, no install needed. Creates a real SQL database from your CSV.
- **matplotlib** — for charts and visualizations.

## Step 1 — Import Libraries
These three lines load the tools we need. Think of it like opening Excel, a calculator, and a charting tool.

In [ ]:
import pandas as pd          # pandas: for loading and manipulating data (like Excel but in code)
import sqlite3               # sqlite3: for creating and querying a SQL database
import matplotlib.pyplot as plt  # matplotlib: for charts
import matplotlib.ticker as mticker

# Make charts look clean
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Libraries loaded successfully.')

## Step 2 — Load the Transaction CSV
`pd.read_csv()` reads a CSV file into a **DataFrame** — the pandas equivalent of a spreadsheet table.  
Every row is a transaction. Every column is a field (date, description, amount, etc.).

In [ ]:
# Load the CSV — change this path to point at your exported file
CSV_FILE = 'sample-data/january-2024.csv'

df = pd.read_csv(CSV_FILE)

# Convert the 'date' column to a proper date type (not just text)
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Convert 'amount' to a number (remove any $ signs or commas first)
df['amount'] = pd.to_numeric(
    df['amount'].astype(str).str.replace(r'[\$,]', '', regex=True),
    errors='coerce'
)

# Drop any rows where amount or description is missing
df = df.dropna(subset=['amount', 'description'])
df = df[df['amount'] != 0]

# Show the first 5 rows — like looking at the top of a spreadsheet
print(f'Loaded {len(df)} transactions')
df.head()

## Step 3 — Load Data into SQLite
Here we create a real SQL database in memory and load the DataFrame into it.  
This lets us run proper SQL queries against the data — exactly as you would in a firm's accounting system.

In [ ]:
# Create an in-memory SQLite database
# ':memory:' means it lives in RAM — fast, no file created
conn = sqlite3.connect(':memory:')

# Load the DataFrame into a SQL table called 'transactions'
# if_exists='replace' means: if the table already exists, replace it
df.to_sql('transactions', conn, if_exists='replace', index=True, index_label='id')

print('Database created. Table: transactions')
print(f'Rows loaded: {len(df)}')

# Quick check — run a simple SQL query to confirm it worked
result = pd.read_sql('SELECT COUNT(*) AS total_rows FROM transactions', conn)
print(result)

---
## Analysis Queries
The following cells each answer a specific accounting question.  
Each one uses a SQL query via `pd.read_sql()`, which runs the SQL and returns the result as a DataFrame.

### Query 1 — Total Spending by Category
**Accounting concept:** Expense classification. The first thing any bookkeeper produces.  
`GROUP BY category` groups all rows with the same category together, then `SUM(amount)` adds them up.

In [ ]:
query = """
SELECT
    category,
    COUNT(*)                    AS num_transactions,
    ROUND(SUM(amount), 2)       AS total_spent,
    ROUND(AVG(amount), 2)       AS avg_transaction,
    ROUND(SUM(amount) * 100.0 /
        (SELECT SUM(amount) FROM transactions), 1) AS pct_of_total
FROM transactions
GROUP BY category
ORDER BY total_spent DESC
"""

by_category = pd.read_sql(query, conn)

# Chart
fig, ax = plt.subplots()
ax.barh(by_category['category'], by_category['total_spent'], color='#111210')
ax.set_xlabel('Total Spent (USD)')
ax.set_title('Total Spending by Category', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

by_category

### Query 2 — Identify Large Transactions (Audit Materiality)
**Accounting concept:** In audit work, auditors focus testing on large transactions because they have the most impact on the financial statements. This is called the **materiality threshold**.  
The `WHERE amount > 1000` filters to only show transactions above $1,000.

In [ ]:
MATERIALITY_THRESHOLD = 1000  # Change this to match the engagement

query = f"""
SELECT
    date,
    description,
    ROUND(amount, 2)   AS amount,
    category,
    flag
FROM transactions
WHERE amount > {MATERIALITY_THRESHOLD}
ORDER BY amount DESC
"""

large_txns = pd.read_sql(query, conn)
print(f'Transactions over ${MATERIALITY_THRESHOLD:,}: {len(large_txns)}')
print(f'Total value: ${large_txns["amount"].sum():,.2f}')
large_txns

### Query 3 — Duplicate Transaction Detection
**Accounting concept:** Duplicate payments are a common internal control failure. This query finds transactions with the same amount and description within 7 days of each other.  
The `pandas` approach here is easier than SQL for this type of self-comparison.

In [ ]:
# Sort by description and date so duplicates are next to each other
df_sorted = df.sort_values(['description', 'amount', 'date']).reset_index(drop=True)

duplicates = []

# Compare each transaction to the next one
for i in range(len(df_sorted) - 1):
    current = df_sorted.iloc[i]
    next_row = df_sorted.iloc[i + 1]

    same_desc   = current['description'] == next_row['description']
    same_amount = current['amount'] == next_row['amount']
    close_date  = abs((next_row['date'] - current['date']).days) <= 7

    if same_desc and same_amount and close_date:
        duplicates.append({
            'description':    current['description'],
            'amount':         current['amount'],
            'first_date':     current['date'].date(),
            'duplicate_date': next_row['date'].date(),
            'days_apart':     abs((next_row['date'] - current['date']).days)
        })

dups_df = pd.DataFrame(duplicates)
print(f'Potential duplicates found: {len(dups_df)}')
if len(dups_df):
    print(f'Total value at risk: ${dups_df["amount"].sum():,.2f}')
    print(dups_df)

### Query 4 — Contractor Payments & 1099 Threshold
**Accounting concept:** Under IRS rules, any contractor paid **$600 or more** in a calendar year must receive a Form 1099-NEC. Missing this is a compliance violation.  
The `CASE WHEN` statement is SQL's version of an if/else — it applies a label based on a condition.

In [ ]:
query = """
SELECT
    description                     AS contractor,
    COUNT(*)                        AS num_payments,
    ROUND(SUM(amount), 2)           AS total_paid,
    CASE
        WHEN SUM(amount) >= 600 THEN 'YES — 1099-NEC Required'
        ELSE 'No — under $600'
    END                             AS form_1099
FROM transactions
WHERE category = 'Contractor / Professional Services'
GROUP BY description
ORDER BY total_paid DESC
"""

contractors = pd.read_sql(query, conn)
print('1099 Compliance Check:')
contractors

### Query 5 — Meals & Entertainment Tax Deductibility
**Accounting concept:** IRS rules limit Meals & Entertainment deductions to **50%**. Tracking the non-deductible portion is important for accurate tax reporting.  
This demonstrates computed columns — creating new values based on existing data.

In [ ]:
query = """
SELECT
    date,
    description,
    ROUND(amount, 2)            AS total_amount,
    ROUND(amount * 0.50, 2)     AS tax_deductible,
    ROUND(amount * 0.50, 2)     AS non_deductible
FROM transactions
WHERE category = 'Meals & Entertainment'
ORDER BY date
"""

meals = pd.read_sql(query, conn)

total_meals      = meals['total_amount'].sum()
total_deductible = meals['tax_deductible'].sum()

print(f'Total M&E spend:         ${total_meals:,.2f}')
print(f'Tax-deductible (50%):    ${total_deductible:,.2f}')
print(f'Non-deductible (50%):    ${total_meals - total_deductible:,.2f}')
meals

### Query 6 — Monthly Expense Trend
**Accounting concept:** Month-over-month comparison is the foundation of management reporting. Finance teams look at this every month-end close.  
`resample('ME')` is a pandas feature that groups data by month automatically.

In [ ]:
# Set date as the index so we can use time-based operations
df_time = df.set_index('date').copy()

# Resample by month-end ('ME'), sum the amounts
monthly = df_time['amount'].resample('ME').agg(['sum', 'count', 'mean'])
monthly.columns = ['total_spent', 'num_transactions', 'avg_transaction']
monthly.index = monthly.index.strftime('%b %Y')

# Chart
fig, ax = plt.subplots()
bars = ax.bar(monthly.index, monthly['total_spent'], color='#1e3a7a', width=0.4)
ax.bar_label(bars, fmt='$%.0f', padding=4, fontsize=9)
ax.set_title('Monthly Expense Total', fontweight='bold')
ax.set_ylabel('Total Expenses (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

monthly

### Query 7 — Audit Sample Selection
**Accounting concept:** Auditors can't test every transaction — there are too many. Instead, they use **audit sampling**:  
1. **Risk-based**: Always select items above the materiality threshold  
2. **Systematic**: Select every Nth item from the remaining population  

This mirrors real audit methodology (CAS 530 / ISA 530 — Audit Sampling).

In [ ]:
MATERIALITY  = 1000   # Test everything above this
SAMPLE_EVERY = 5      # Test every 5th item below materiality

df_reset = df.reset_index(drop=True)

def select_audit_sample(row, idx):
    if row['amount'] >= MATERIALITY:
        return 'SELECTED — Above materiality'
    elif idx % SAMPLE_EVERY == 0:
        return f'SELECTED — Systematic (every {SAMPLE_EVERY}th)'
    else:
        return 'Not selected'

df_reset['audit_selection'] = [
    select_audit_sample(row, i) for i, row in df_reset.iterrows()
]

selected = df_reset[df_reset['audit_selection'].str.startswith('SELECTED')]

print(f'Total transactions:   {len(df_reset)}')
print(f'Selected for testing: {len(selected)}')
print(f'Sample coverage:      {len(selected)/len(df_reset)*100:.1f}%')
print(f'Value coverage:       {selected["amount"].sum()/df_reset["amount"].sum()*100:.1f}% of total spend')

selected[['date','description','amount','category','audit_selection']]

### Query 8 — Accounts Payable Aging
**Accounting concept:** AP aging buckets unpaid invoices by how long they've been outstanding.  
It's one of the most common reports in any accounting department and directly informs cash flow management.  
`pd.cut()` is the pandas way to sort values into buckets — equivalent to nested IF statements in Excel.

In [ ]:
from datetime import date

today = pd.Timestamp.today()

df_ap = df.copy()
df_ap['days_outstanding'] = (today - df_ap['date']).dt.days

# Bucket into aging ranges using pd.cut()
# This is equivalent to a nested IF in Excel
bins   = [0, 30, 60, 90, float('inf')]
labels = ['0-30 days', '31-60 days', '61-90 days', 'Over 90 days']
df_ap['aging_bucket'] = pd.cut(df_ap['days_outstanding'], bins=bins, labels=labels, right=True)

aging_summary = df_ap.groupby('aging_bucket', observed=True)['amount'].agg(['sum','count'])
aging_summary.columns = ['total_outstanding', 'num_invoices']

# Chart
colors = ['#1e6640', '#a87c1a', '#c13820', '#6b1010']
fig, ax = plt.subplots()
bars = ax.bar(aging_summary.index, aging_summary['total_outstanding'], color=colors)
ax.bar_label(bars, fmt='$%.0f', padding=4, fontsize=9)
ax.set_title('Accounts Payable Aging', fontweight='bold')
ax.set_ylabel('Amount Outstanding (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

aging_summary

### Query 9 — Category Variance Analysis (Budget vs Actual)
**Accounting concept:** Variance analysis compares what was budgeted to what was actually spent. This is a core part of month-end close and management reporting at every firm.  
`merge()` in pandas is equivalent to a SQL JOIN — it combines two tables on a common key.

In [ ]:
# Define monthly budgets (what was planned)
# In a real system this would come from a budgets table in the database
budgets = pd.DataFrame([
    {'category': 'Payroll',                          'budget': 38000},
    {'category': 'Rent & Facilities',                'budget': 3500},
    {'category': 'Advertising & Marketing',          'budget': 5000},
    {'category': 'Software & Subscriptions',         'budget': 2500},
    {'category': 'Contractor / Professional Services','budget': 8000},
    {'category': 'Meals & Entertainment',            'budget': 600},
    {'category': 'Travel',                           'budget': 1500},
    {'category': 'Utilities & Hosting',              'budget': 600},
])

# Calculate actual spending per category
actuals = df.groupby('category')['amount'].sum().reset_index()
actuals.columns = ['category', 'actual']

# JOIN budgets to actuals on category name
variance = pd.merge(budgets, actuals, on='category', how='left').fillna(0)
variance['variance']      = variance['actual'] - variance['budget']
variance['variance_pct']  = ((variance['actual'] - variance['budget']) / variance['budget'] * 100).round(1)
variance['status']        = variance['variance'].apply(
    lambda v: '🔴 Over budget' if v > 0 else '🟢 Under budget'
)

variance = variance.sort_values('variance', ascending=False)
print('Budget vs Actual — Variance Analysis:')
variance

### Query 10 — Executive Summary Report
A clean summary of all key metrics — the kind of one-page output you'd present to a partner or CFO.

In [ ]:
total_spend     = df['amount'].sum()
num_txns        = len(df)
largest_txn     = df.loc[df['amount'].idxmax()]
top_category    = df.groupby('category')['amount'].sum().idxmax()
top_cat_pct     = df.groupby('category')['amount'].sum().max() / total_spend * 100
flagged         = df[df['flag'] != 'none']
over_1000       = df[df['amount'] > 1000]
payroll_total   = df[df['category'] == 'Payroll']['amount'].sum()

print('=' * 55)
print('  LEDGERAI — EXECUTIVE SUMMARY REPORT')
print(f'  Period: {df["date"].min().strftime("%B %Y")} — {df["date"].max().strftime("%B %Y")}')
print('=' * 55)
print(f'  Total Expenses:          ${total_spend:>12,.2f}')
print(f'  Total Transactions:      {num_txns:>12,}')
print(f'  Avg Transaction Size:    ${total_spend/num_txns:>12,.2f}')
print('-' * 55)
print(f'  Largest Transaction:     ${largest_txn["amount"]:>12,.2f}')
print(f'    → {largest_txn["description"][:42]}')
print(f'  Top Expense Category:    {top_category}')
print(f'    → {top_cat_pct:.1f}% of total spend')
print('-' * 55)
print(f'  Payroll Total:           ${payroll_total:>12,.2f}')
print(f'  Payroll % of Expenses:   {payroll_total/total_spend*100:>11.1f}%')
print(f'  Transactions > $1,000:   {len(over_1000):>12,}')
print(f'  Flagged for Review:      {len(flagged):>12,}')
print(f'  Potential Duplicates:    {len(dups_df):>12,}')
print('=' * 55)

---
## How This Connects to LedgerAI

Everything in this notebook mirrors what the LedgerAI web app does visually:

| This Notebook | LedgerAI Web App |
|---|---|
| `pd.read_csv()` | File upload & CSV parser |
| SQLite `transactions` table | In-memory transaction state |
| Query 1 — Category totals | Dashboard → Category Breakdown |
| Query 3 — Duplicate detection | Reports → Duplicate Detector |
| Query 4 — 1099 contractor check | Flagged Items |
| Query 5 — M&E 50% rule | Flagged Items |
| Query 7 — Audit sampling | Reports → Audit Trail |
| Query 8 — AP aging | Reports → Reconciliation |
| Query 9 — Budget vs Actual | Reports → Budget vs Actual |

**The web app is the user-facing interface. This notebook is the data engineering layer underneath it.**  
In a real accounting firm, analysts use Python/SQL pipelines like this to process data at scale before it surfaces in a UI.

---
*Built by Zohaib Ghani — BBA Accounting, Wilfrid Laurier University*  
*zohaibghani556@gmail.com*